# Data Research

Цей ноутбук виконує дослідження даних:
- Підключається до БД MySQL
- Обчислює базові статистики
- Виконує TF-IDF аналіз та кластеризацію
- Зберігає JSON-звіт у `/shared/reports/data_research_report.json`

In [ ]:
import os
import json
import pandas as pd
import numpy as np
from sqlalchemy import create_engine
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans

db_host = os.environ.get('MYSQL_HOST', 'db')
db_user = os.environ.get('MYSQL_USER', 'appuser')
db_password = os.environ.get('MYSQL_PASSWORD', 'apppassword')
db_name = os.environ.get('MYSQL_DATABASE', 'docflow')

connection_string = f'mysql+pymysql://{db_user}:{db_password}@{db_host}:3306/{db_name}'
engine = create_engine(connection_string)

df = pd.read_sql('SELECT * FROM documents', engine)
print(f'[data_research] Завантажено {len(df)} записів')

In [ ]:
total_records = len(df)
unique_recipients = df['Адресати'].nunique() if 'Адресати' in df.columns else 0

if 'Адресати' in df.columns:
    top_recipients = df['Адресати'].value_counts().head(10).to_dict()
else:
    top_recipients = {}

print(f'Загальна кількість документів: {total_records}')
print(f'Кількість унікальних отримувачів: {unique_recipients}')
print(f'\nТоп-10 отримувачів:')
for name, count in top_recipients.items():
    print(f'  {name}: {count}')

In [ ]:
text_col = None
for col in df.columns:
    if 'зміст' in col.lower() or 'короткий' in col.lower():
        text_col = col
        break

cluster_info = {}
if text_col:
    df[text_col] = df[text_col].fillna('немає змісту').astype(str)
    vectorizer = TfidfVectorizer(max_features=500, stop_words=None)
    X = vectorizer.fit_transform(df[text_col])

    num_clusters = 4
    kmeans = KMeans(n_clusters=num_clusters, random_state=42, n_init=10)
    df['cluster'] = kmeans.fit_predict(X)

    cluster_distribution = df['cluster'].value_counts().sort_index().to_dict()
    cluster_distribution = {str(k): int(v) for k, v in cluster_distribution.items()}

    terms = vectorizer.get_feature_names_out()
    order_centroids = kmeans.cluster_centers_.argsort()[:, ::-1]
    cluster_keywords = {}
    for i in range(num_clusters):
        keywords = [terms[ind] for ind in order_centroids[i, :5]]
        cluster_keywords[str(i)] = keywords

    cluster_info = {
        'num_clusters': num_clusters,
        'distribution': cluster_distribution,
        'keywords': cluster_keywords
    }

    print(f'\nКластеризація ({num_clusters} кластерів):')
    for cid, count in cluster_distribution.items():
        kw = ', '.join(cluster_keywords[cid])
        print(f'  Кластер {cid}: {count} документів — ключові слова: {kw}')
else:
    print('Текстовий стовпець не знайдено.')

In [ ]:
date_col = None
for col in df.columns:
    if 'дата' in col.lower() or 'реєстрац' in col.lower():
        date_col = col
        break

date_stats = {}
if date_col:
    try:
        df[date_col] = pd.to_datetime(df[date_col])
        docs_per_day = df.groupby(df[date_col].dt.day).size()
        date_stats = {
            'min_date': str(df[date_col].min()),
            'max_date': str(df[date_col].max()),
            'mean_docs_per_day': round(float(docs_per_day.mean()), 2),
            'max_docs_per_day': int(docs_per_day.max()),
            'min_docs_per_day': int(docs_per_day.min()),
            'docs_per_day': {str(k): int(v) for k, v in docs_per_day.to_dict().items()}
        }
        print(f'\nСтатистика по датах:')
        print(f'  Період: {date_stats["min_date"]} — {date_stats["max_date"]}')
        print(f'  Середня кількість документів на день: {date_stats["mean_docs_per_day"]}')
        print(f'  Максимум: {date_stats["max_docs_per_day"]}, Мінімум: {date_stats["min_docs_per_day"]}')
    except Exception as e:
        print(f'Помилка обробки дат: {e}')

In [ ]:
report = {
    'total_records': total_records,
    'unique_recipients': unique_recipients,
    'top_recipients': top_recipients,
    'clustering': cluster_info,
    'date_statistics': date_stats
}

report_path = '/shared/reports/data_research_report.json'
os.makedirs(os.path.dirname(report_path), exist_ok=True)

with open(report_path, 'w', encoding='utf-8') as f:
    json.dump(report, f, ensure_ascii=False, indent=2)

print(f'\nЗвіт збережено: {report_path}')